# Single Tree Reconstruction Pipeline
AdTree-based reconstruction for individual tree point clouds.

**Input:** single tree point cloud file (.las, .laz, .ply, .txt, .csv)

**Output:**
- Point cloud (.ply)
- Skeleton (.ply)
- Branches (.obj)
- Leaves (.obj, filtered versions)

**Required files to upload:**
- Your single tree point cloud file (.las / .laz / .ply / .txt / .csv)
- `AdTree-main.zip`


## Step 1: Install dependencies

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'laspy[lazrs]', 'open3d', 'numpy'], check=True)

print('Installation complete.')


## Step 2: System dependencies + compile AdTree
> Takes 3-5 minutes.

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq \
    cmake build-essential \
    libboost-all-dev \
    libgl1-mesa-dev libglu1-mesa-dev \
    libxrandr-dev libxinerama-dev libxcursor-dev libxi-dev libxext-dev

rm -rf /content/AdTree_build/
mkdir -p /content/AdTree_build/
unzip /content/AdTree-main.zip -d /content/AdTree_build/ 2>&1 | tail -3

SRCDIR=/content/AdTree_build/AdTree-main

# OpenGL fix
find $SRCDIR -name "CMakeLists.txt" -exec \
    sed -i 's/OpenGL::OpenGL/OpenGL::GL/g' {} \;

sed -i 's/target_link_libraries(${PROJECT_NAME} PRIVATE easy3d_algo/target_link_libraries(${PROJECT_NAME} PRIVATE GLdispatch easy3d_algo/' \
    $SRCDIR/AdTree/CMakeLists.txt
ln -sf /usr/lib/x86_64-linux-gnu/libGLdispatch.so.0 \
       /usr/lib/x86_64-linux-gnu/libGLdispatch.so 2>/dev/null || true

# Apply all patches to skeleton.cpp
python3 << 'PYEOF'
SKEL = '/content/AdTree_build/AdTree-main/AdTree/skeleton.cpp'
with open(SKEL, 'r') as f:
    content = f.read()

# Patch A: leaf density and size
content = content.replace(
    'int density = ceil(random_float() * 10);',
    'int density = ceil(random_float() * 1);'
)
content = content.replace(
    'generate_leaves(currentLeafVertex, 0.05);',
    'generate_leaves(currentLeafVertex, 0.02);'
)
content = content.replace(
    'double radius = 0.2 / log((float)num_edges(simplified_skeleton_));',
    'double radius = 0.04 / log((float)num_edges(simplified_skeleton_));'
)
print('Patch A: leaf density reduced')

# Patch epsiony: 2% -> 10%
content = content.replace(
    '\tdouble epsiony = 0.02;\n',
    '\tdouble epsiony = 0.10;\n'
)
print('Patch epsiony: trunk list at 10%')

# Patch C: leaf base attached to branch tip
old_generate = (
    'void Skeleton::generate_leaves(SGraphVertexDescriptor i_LeafVertex, double leafsize_Factor)\n'
    '{\n'
    '\t//generate a random density number\n'
    '    int density = ceil(random_float() * 1);\n'
    '    double radius = 0.04 / log((float)num_edges(simplified_skeleton_));\n'
    '\t//get the position of the current leaf vertex and its parent\n'
    '    vec3 pCurrent = simplified_skeleton_[i_LeafVertex].cVert;\n'
    '    SGraphVertexDescriptor i_LeafParent = simplified_skeleton_[i_LeafVertex].nParent;\n'
    '    vec3 pParent = simplified_skeleton_[i_LeafParent].cVert;\n'
    '\t//get the end position where the leaf should grow\n'
    '    vec3 pEnd = pCurrent - (random_float() / 2.0) * ((pCurrent - pParent).normalize());\n'
    '\n'
    '\t//generate i-th random leaf\n'
    '\tfor (int i = 0; i < density; ++i)\n'
    '\t{\n'
    '\t\t//generate a random leaf position\n'
    '        vec3 dirLeaf((random_float() - 0.5) / 0.5, (random_float() - 0.5) / 0.5, (random_float() - 0.5) / 0.5);\n'
    '\t\tdirLeaf = dirLeaf.normalize();\n'
    '        double l = random_float() * radius;\n'
    '\t\tvec3 pLeaf = pEnd + dirLeaf * l;\n'
    '\t\t//generate normal and color vector\n'
    '\t\tvec3 dirParent2Leaf = (pLeaf - pParent).normalize();\n'
    '\t\tvec3 normal = (cross(dirParent2Leaf, dirLeaf)).normalize();\n'
    '\t\t//generate a new leaf\n'
    '\t\tLeaf newleaf;\n'
    '\t\tnewleaf.cPos = pLeaf;\n'
    '\t\tnewleaf.cDir = dirLeaf;\n'
    '\t\t//generate a random normal vector direction\n'
    '        vec3 delta((random_float() - 0.5) / 0.5, (random_float() - 0.5) / 0.5, (random_float() - 0.5) / 0.5);\n'
    '        newleaf.cNormal = (normal + random_float()*delta*0.5).normalize();\n'
    '\t\tnewleaf.pSource = i_LeafVertex;\n'
    '\t\tnewleaf.nLength = BoundingDistance_ * leafsize_Factor;\n'
    '\t\tnewleaf.nRad = newleaf.nLength / 5;\n'
    '\t\tVecLeaves_.push_back(newleaf);\n'
    '\t}\n'
    '\n'
    '\treturn;\n'
    '}'
)
new_generate = (
    'void Skeleton::generate_leaves(SGraphVertexDescriptor i_LeafVertex, double leafsize_Factor)\n'
    '{\n'
    '\t//generate a random density number\n'
    '    int density = ceil(random_float() * 1);\n'
    '    double radius = 0.04 / log((float)num_edges(simplified_skeleton_));\n'
    '\t//get the position of the current leaf vertex and its parent\n'
    '    vec3 pCurrent = simplified_skeleton_[i_LeafVertex].cVert;\n'
    '    SGraphVertexDescriptor i_LeafParent = simplified_skeleton_[i_LeafVertex].nParent;\n'
    '    vec3 pParent = simplified_skeleton_[i_LeafParent].cVert;\n'
    '\t// branch growth direction\n'
    '    vec3 branchDir = (pCurrent - pParent).normalize();\n'
    '\n'
    '\t//generate i-th random leaf\n'
    '\tfor (int i = 0; i < density; ++i)\n'
    '\t{\n'
    '\t\t// leaf base directly at branch tip\n'
    '        double offset = random_float() * radius * 0.5;\n'
    '        vec3 pLeaf = pCurrent - branchDir * offset;\n'
    '\n'
    '\t\t// leaf direction: perpendicular away from branch\n'
    '        vec3 randPerp((random_float()-0.5)/0.5, (random_float()-0.5)/0.5, (random_float()-0.5)/0.5);\n'
    '        randPerp = randPerp.normalize();\n'
    '        randPerp = (randPerp - branchDir * dot(randPerp, branchDir)).normalize();\n'
    '        vec3 dirLeaf = (randPerp * 0.6f + branchDir * 0.4f).normalize();\n'
    '\n'
    '\t\t//generate normal and color vector\n'
    '\t\tvec3 dirParent2Leaf = (pLeaf - pParent).normalize();\n'
    '\t\tvec3 normal = (cross(dirParent2Leaf, dirLeaf)).normalize();\n'
    '\t\t//generate a new leaf\n'
    '\t\tLeaf newleaf;\n'
    '\t\tnewleaf.cPos = pLeaf;\n'
    '\t\tnewleaf.cDir = dirLeaf;\n'
    '\t\t//generate a random normal vector direction\n'
    '        vec3 delta((random_float()-0.5)/0.5, (random_float()-0.5)/0.5, (random_float()-0.5)/0.5);\n'
    '        newleaf.cNormal = (normal + random_float()*delta*0.5).normalize();\n'
    '\t\tnewleaf.pSource = i_LeafVertex;\n'
    '\t\tnewleaf.nLength = BoundingDistance_ * leafsize_Factor;\n'
    '\t\tnewleaf.nRad = newleaf.nLength / 5;\n'
    '\t\tVecLeaves_.push_back(newleaf);\n'
    '\t}\n'
    '\n'
    '\treturn;\n'
    '}'
)
assert old_generate in content, 'ERROR: generate_leaves not found!'
content = content.replace(old_generate, new_generate, 1)
print('Patch C: leaf base attached to branch tip')

# Patch B: elliptic leaf shape
old_func = (
    'bool Skeleton::reconstruct_leaves(SurfaceMesh *mesh) {\n'
    '    if (!add_leaves())\n'
    '        return false;\n'
    '\n'
    '    if (VecLeaves_.empty())\n'
    '        return false;\n'
    '\n'
    '    for (std::size_t i = 0; i < VecLeaves_.size(); i++) {\n'
    '        const Leaf& iLeaf = VecLeaves_[i];\n'
    '        //compute the center and major axis, minor axis of the leaf quad\n'
    '        vec3 pCenter((iLeaf.cPos + (0.5 * iLeaf.cDir * iLeaf.nRad)));\n'
    '        vec3 dirMajor(0.5 * iLeaf.cDir * iLeaf.nLength);\n'
    '        vec3 dirMinor(0.5 * cross(iLeaf.cDir, iLeaf.cNormal)*iLeaf.nRad);\n'
    '        //compute the corner coordinates\n'
    '        const vec3 a = pCenter - dirMajor - dirMinor;\n'
    '        const vec3 b = pCenter + dirMajor - dirMinor;\n'
    '        const vec3 c = pCenter + dirMajor + dirMinor;\n'
    '        const vec3 d = pCenter - dirMajor + dirMinor;\n'
    '        SurfaceMesh::Vertex va = mesh->add_vertex(a);\n'
    '        SurfaceMesh::Vertex vb = mesh->add_vertex(b);\n'
    '        SurfaceMesh::Vertex vc = mesh->add_vertex(c);\n'
    '        SurfaceMesh::Vertex vd = mesh->add_vertex(d);\n'
    '        mesh->add_triangle(va, vb, vc);\n'
    '        mesh->add_triangle(va, vc, vd);\n'
    '    }\n'
    '\n'
    '    return true;\n'
    '}'
)
new_func = (
    'bool Skeleton::reconstruct_leaves(SurfaceMesh *mesh) {\n'
    '    if (!add_leaves())\n'
    '        return false;\n'
    '\n'
    '    if (VecLeaves_.empty())\n'
    '        return false;\n'
    '\n'
    '    const int nSegs = 6;\n'
    '    for (std::size_t i = 0; i < VecLeaves_.size(); i++) {\n'
    '        const Leaf& iLeaf = VecLeaves_[i];\n'
    '        vec3 dir = iLeaf.cDir; vec3 norm = iLeaf.cNormal;\n'
    '        vec3 axisL = dir.normalize() * iLeaf.nLength;\n'
    '        vec3 axisW = cross(dir, norm).normalize() * iLeaf.nRad;\n'
    '        vec3 pBase = iLeaf.cPos;\n'
    '        std::vector<SurfaceMesh::Vertex> leftVerts, rightVerts;\n'
    '        for (int s = 0; s <= nSegs; ++s) {\n'
    '            double t = (double)s / nSegs;\n'
    '            double width = sin(M_PI * t) * (2.0 - 0.3 * t);\n'
    '            vec3 pAlong = pBase + axisL * t;\n'
    '            leftVerts.push_back(mesh->add_vertex(pAlong - axisW * width * 0.5));\n'
    '            rightVerts.push_back(mesh->add_vertex(pAlong + axisW * width * 0.5));\n'
    '        }\n'
    '        for (int s = 0; s < nSegs; ++s) {\n'
    '            mesh->add_triangle(leftVerts[s],   rightVerts[s],   rightVerts[s+1]);\n'
    '            mesh->add_triangle(leftVerts[s],   rightVerts[s+1], leftVerts[s+1]);\n'
    '        }\n'
    '    }\n'
    '    return true;\n'
    '}'
)
assert old_func in content, 'ERROR: reconstruct_leaves not found!'
content = content.replace(old_func, new_func, 1)
print('Patch B: elliptic leaf shape applied')

# Patch D: Least-Squares circle fit for trunk radius
old_bbox = (
    '\t//project the trunk points on the xy plane and get the bounding box\n'
    '\tdouble minX = DBL_MAX;\n'
    '\tdouble maxX = -DBL_MAX;\n'
    '\tdouble minY = DBL_MAX;\n'
    '\tdouble maxY = -DBL_MAX;\n'
    '\tfor (int nP = 0; nP < trunkList.size(); nP++)\n'
    '\t{\n'
    '\t\tif (minX > trunkList[nP].x)\n'
    '\t\t\tminX = trunkList[nP].x;\n'
    '\t\tif (maxX < trunkList[nP].x)\n'
    '\t\t\tmaxX = trunkList[nP].x;\n'
    '\t\tif (minY > trunkList[nP].y)\n'
    '\t\t\tminY = trunkList[nP].y;\n'
    '\t\tif (maxY < trunkList[nP].y)\n'
    '\t\t\tmaxY = trunkList[nP].y;\n'
    '\t}\n'
    '\n'
    '\t//assign the raw radius value and return\n'
    '    TrunkRadius_ = std::max((maxX - minX), (maxY - minY)) / 2.0;\n'
)
new_bbox = (
    '\t// Patch D: Least-Squares circle fit instead of bounding box\n'
    '\tdouble minX = DBL_MAX, maxX = -DBL_MAX;\n'
    '\tdouble minY = DBL_MAX, maxY = -DBL_MAX;\n'
    '\tfor (int nP = 0; nP < (int)trunkList.size(); nP++) {\n'
    '\t\tif (minX > trunkList[nP].x) minX = trunkList[nP].x;\n'
    '\t\tif (maxX < trunkList[nP].x) maxX = trunkList[nP].x;\n'
    '\t\tif (minY > trunkList[nP].y) minY = trunkList[nP].y;\n'
    '\t\tif (maxY < trunkList[nP].y) maxY = trunkList[nP].y;\n'
    '\t}\n'
    '\tdouble cx = 0.0, cy = 0.0;\n'
    '\tfor (int nP = 0; nP < (int)trunkList.size(); nP++) {\n'
    '\t\tcx += trunkList[nP].x; cy += trunkList[nP].y;\n'
    '\t}\n'
    '\tcx /= (int)trunkList.size(); cy /= (int)trunkList.size();\n'
    '\tdouble r  = std::max((maxX - minX), (maxY - minY)) / 2.0;\n'
    '\tdouble r_max = r * 2.0;\n'
    '\tif ((int)trunkList.size() >= 1000) {\n'
    '\t\tfor (int iter = 0; iter < 100; ++iter) {\n'
    '\t\t\tdouble dCx=0, dCy=0, dR=0;\n'
    '\t\t\tdouble A00=0,A01=0,A02=0,A11=0,A12=0,A22=0,b0=0,b1=0,b2=0;\n'
    '\t\t\tfor (int nP = 0; nP < (int)trunkList.size(); nP++) {\n'
    '\t\t\t\tdouble dx=trunkList[nP].x-cx, dy=trunkList[nP].y-cy;\n'
    '\t\t\t\tdouble dist=std::sqrt(dx*dx+dy*dy);\n'
    '\t\t\t\tif (dist<1e-10) continue;\n'
    '\t\t\t\tdouble res=dist-r, Jcx=-dx/dist, Jcy=-dy/dist, Jr=-1.0;\n'
    '\t\t\t\tA00+=Jcx*Jcx; A01+=Jcx*Jcy; A02+=Jcx*Jr;\n'
    '\t\t\t\tA11+=Jcy*Jcy; A12+=Jcy*Jr;  A22+=Jr*Jr;\n'
    '\t\t\t\tb0+=Jcx*res;  b1+=Jcy*res;  b2+=Jr*res;\n'
    '\t\t\t}\n'
    '\t\t\tdouble det=A00*(A11*A22-A12*A12)-A01*(A01*A22-A12*A02)+A02*(A01*A12-A11*A02);\n'
    '\t\t\tif (std::abs(det)<1e-14) break;\n'
    '\t\t\tdCx=(b0*(A11*A22-A12*A12)-A01*(b1*A22-A12*b2)+A02*(b1*A12-A11*b2))/det;\n'
    '\t\t\tdCy=(A00*(b1*A22-A12*b2)-b0*(A01*A22-A12*A02)+A02*(A01*b2-b1*A02))/det;\n'
    '\t\t\tdR=(A00*(A11*b2-b1*A12)-A01*(A01*b2-b1*A02)+b0*(A01*A12-A11*A02))/det;\n'
    '\t\t\tcx-=dCx; cy-=dCy; r-=dR;\n'
    '\t\t\tif (r<0) r=std::abs(r);\n'
    '\t\t\tif (r>r_max) { r=r_max; break; }\n'
    '\t\t\tif (std::abs(dCx)+std::abs(dCy)+std::abs(dR)<1e-8) break;\n'
    '\t\t}\n'
    '\t}\n'
    '    TrunkRadius_ = r;\n'
)
assert old_bbox in content, 'ERROR: BBox not found!'
content = content.replace(old_bbox, new_bbox, 1)
print('Patch D: least-squares circle fit applied')

with open(SKEL, 'w') as f:
    f.write(content)
print('All patches applied successfully')
PYEOF

mkdir -p $SRCDIR/Release && cd $SRCDIR/Release
cmake -DCMAKE_BUILD_TYPE=Release -DCMAKE_CXX_FLAGS="-w" .. 2>&1 | tail -4
echo "Compiling AdTree (3-5 min)..."
make -j$(nproc) 2>&1 | tail -10

EXEC=$(find $SRCDIR/Release/bin -name 'AdTree' -type f 2>/dev/null | head -1)
if [ -n "$EXEC" ]; then
    echo "AdTree compiled: $EXEC"
    echo "$EXEC" > /content/adtree_exec_path.txt
else
    echo "ERROR: compilation failed"
    make -j$(nproc) 2>&1 | grep 'error:' | head -5
fi


## Step 3: Configuration
Set filtering options here.

In [ ]:
import os, glob

# --- Input file ---
# The pipeline auto-detects your uploaded file.
# Supported: .las .laz .ply .txt .csv
# To set manually: INPUT_FILE = '/content/your_file.laz'
EXTENSIONS = ['*.las', '*.laz', '*.ply', '*.txt', '*.csv']
found = []
for ext in EXTENSIONS:
    found += glob.glob(f'/content/{ext}')
found = [f for f in found if not os.path.basename(f).startswith('.')]
assert found, (
    'No point cloud file found. '
    'Upload a .las / .laz / .ply / .txt / .csv file to Colab.'
)
INPUT_FILE = found[0]
if len(found) > 1:
    print(f'Multiple files found - using: {os.path.basename(INPUT_FILE)}')
    print(f'All found: {[os.path.basename(f) for f in found]}')
    print('To use a different file: INPUT_FILE = "/content/your_file.laz"')
else:
    print(f'Input file: {INPUT_FILE}')

TREE_ID  = os.path.splitext(os.path.basename(INPUT_FILE))[0]
OUTPUT_DIR = '/content/tree_output'
FINAL_ZIP  = f'/content/{TREE_ID}_results.zip'

# --- Noise filtering (optional) ---
ENABLE_NOISE_FILTER = False
NB_NEIGHBORS        = 20
STD_RATIO           = 2.0
VOXEL_SIZE          = 0.05   # meters

# --- Leaf filtering ---
FACES_PER_LEAF = 12          # 6 segments x 2 triangles (elliptic leaf shape)
KEEP_RATIOS    = [0.6, 0.3]  # two filtered versions

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Configuration done.')
print(f'  Tree ID    : {TREE_ID}')
print(f'  Output dir : {OUTPUT_DIR}')


## Step 4: Load point cloud

In [ ]:
import laspy
import open3d as o3d
import numpy as np

ext = os.path.splitext(INPUT_FILE)[1].lower()

if ext in ('.las', '.laz'):
    las = laspy.read(INPUT_FILE)
    pts = np.vstack([las.x, las.y, las.z]).T.astype(np.float64)
elif ext == '.ply':
    pcd = o3d.io.read_point_cloud(INPUT_FILE)
    pts = np.asarray(pcd.points).astype(np.float64)
elif ext in ('.txt', '.csv'):
    pts = np.loadtxt(INPUT_FILE, delimiter=None)[:, :3].astype(np.float64)
else:
    raise ValueError(f'Unsupported format: {ext}')

print(f'{len(pts):,} points loaded')
print(f'  X range : {pts[:,0].min():.2f} - {pts[:,0].max():.2f}')
print(f'  Y range : {pts[:,1].min():.2f} - {pts[:,1].max():.2f}')
print(f'  Z range : {pts[:,2].min():.2f} - {pts[:,2].max():.2f}')

z_range = pts[:,2].max() - pts[:,2].min()
xy_range = max(pts[:,0].max()-pts[:,0].min(), pts[:,1].max()-pts[:,1].min())
if z_range < xy_range * 0.3:
    print('WARNING: Z range is very small relative to XY - check that Z points upward')


## Step 5: Noise filtering (optional)
Only runs if `ENABLE_NOISE_FILTER = True` in Step 3.

In [ ]:
if ENABLE_NOISE_FILTER:
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    pcd, _ = pcd.remove_statistical_outlier(
        nb_neighbors=NB_NEIGHBORS, std_ratio=STD_RATIO)
    pcd = pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)
    pts = np.asarray(pcd.points).astype(np.float64)
    print(f'After filtering: {len(pts):,} points')
else:
    print('Noise filtering disabled - skipped.')


## Step 6: AdTree branch and leaf reconstruction

In [ ]:
import subprocess, shutil

with open('/content/adtree_exec_path.txt') as f:
    ADTREE_BIN = f.read().strip()
print(f'AdTree binary: {ADTREE_BIN}')

# Save point cloud as PLY
pcd_save = o3d.geometry.PointCloud()
pcd_save.points = o3d.utility.Vector3dVector(pts)
ply_path = os.path.join(OUTPUT_DIR, f'{TREE_ID}_pointcloud.ply')
o3d.io.write_point_cloud(ply_path, pcd_save)
print(f'Point cloud saved: {ply_path}')

# Write XYZ for AdTree
xyz_file = os.path.join(OUTPUT_DIR, f'{TREE_ID}.xyz')
np.savetxt(xyz_file, pts, fmt='%.6f')

# Run AdTree
adtree_raw = os.path.join(OUTPUT_DIR, 'adtree_raw')
os.makedirs(adtree_raw, exist_ok=True)

print(f'Running AdTree on {len(pts):,} points...')
proc = subprocess.run(
    [ADTREE_BIN, xyz_file, adtree_raw, '-skeleton'],
    capture_output=True, text=True, timeout=900
)

src_br = os.path.join(adtree_raw, f'{TREE_ID}_branches.obj')
src_lv = os.path.join(adtree_raw, f'{TREE_ID}_leaves.obj')
src_sk = os.path.join(adtree_raw, f'{TREE_ID}_skeleton.ply')

assert os.path.exists(src_br), (
    f'AdTree failed - branches.obj not found.\nstderr: {proc.stderr[-500:]}'
)

shutil.copy(src_br, os.path.join(OUTPUT_DIR, f'{TREE_ID}_branches.obj'))
shutil.copy(src_sk, os.path.join(OUTPUT_DIR, f'{TREE_ID}_skeleton.ply'))
shutil.copy(src_lv, os.path.join(OUTPUT_DIR, f'{TREE_ID}_leaves.obj'))

print('AdTree reconstruction complete.')
print(f'  Branches : {OUTPUT_DIR}/{TREE_ID}_branches.obj')
print(f'  Leaves   : {OUTPUT_DIR}/{TREE_ID}_leaves.obj')
print(f'  Skeleton : {OUTPUT_DIR}/{TREE_ID}_skeleton.ply')


## Step 7: Filter leaves

In [ ]:
def filter_leaves_obj(leaves_in, leaves_out, keep_ratio, faces_per_leaf=12):
    """Filter a leaves OBJ file to keep_ratio fraction of leaves."""
    with open(leaves_in) as f:
        lines = f.readlines()
    vertices = [l for l in lines if l.startswith('v ')]
    faces    = [l for l in lines if l.startswith('f ')]
    header   = [l for l in lines if not l.startswith('v ') and not l.startswith('f ')]
    n_leaves = len(faces) // faces_per_leaf
    n_keep   = max(1, int(n_leaves * keep_ratio))
    np.random.seed(42)
    keep_idx = np.sort(np.random.choice(n_leaves, n_keep, replace=False))
    kept_faces = []
    for idx in keep_idx:
        kept_faces.extend(faces[idx*faces_per_leaf : (idx+1)*faces_per_leaf])
    used_v = set()
    for face in kept_faces:
        for token in face.strip().split()[1:]:
            used_v.add(int(token.split('/')[0]) - 1)
    old_to_new = {}
    new_verts  = []
    for old_idx in sorted(used_v):
        old_to_new[old_idx] = len(new_verts) + 1
        new_verts.append(vertices[old_idx])
    new_faces = []
    for face in kept_faces:
        parts   = face.strip().split()
        new_idx = [str(old_to_new[int(p.split('/')[0]) - 1]) for p in parts[1:]]
        new_faces.append('f ' + ' '.join(new_idx) + '\n')
    with open(leaves_out, 'w') as f:
        f.writelines(header)
        f.writelines(new_verts)
        f.writelines(new_faces)

leaves_orig = os.path.join(OUTPUT_DIR, f'{TREE_ID}_leaves.obj')
for ratio in KEEP_RATIOS:
    ratio_str = str(ratio).replace('.', '')
    out_path  = os.path.join(OUTPUT_DIR, f'{TREE_ID}_leaves_filtered{ratio_str}.obj')
    filter_leaves_obj(leaves_orig, out_path, ratio, FACES_PER_LEAF)
    size_kb = os.path.getsize(out_path) / 1024
    print(f'  Leaves filtered {int(ratio*100)}% : {out_path}  ({size_kb:.0f} KB)')


## Step 8: Create output ZIP

In [ ]:
import zipfile

print(f'Creating ZIP: {FINAL_ZIP}')

with zipfile.ZipFile(FINAL_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname   = os.path.relpath(full_path, OUTPUT_DIR)
            zf.write(full_path, arcname)

size_mb = os.path.getsize(FINAL_ZIP) / 1024 / 1024
print(f'ZIP created: {FINAL_ZIP}  ({size_mb:.1f} MB)')

with zipfile.ZipFile(FINAL_ZIP) as zf:
    for name in sorted(zf.namelist()):
        if not name.startswith('adtree_raw'):
            print(f'  {name}')


## Step 9: Download results

In [ ]:
from google.colab import files
files.download(FINAL_ZIP)
print(f'Download started: {os.path.basename(FINAL_ZIP)}')
